<a href="https://colab.research.google.com/github/Dani2003/paper-implementations/blob/main/implementation_of_MELDER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
# [CELL 1 - Dependencies & NLTK Setup]
!pip install -q mediapipe pyctcdecode nltk torch torchvision

import os
import time
import math
import sys
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import deque
from typing import List, Tuple

# Download and verify NLTK words corpus explicitly
import nltk
nltk.download('words', quiet=False)

try:
    from nltk.corpus import words
    VOCAB_WORDS = set(words.words())
except Exception as e:
    print(f"NLTK corpus load warning: {e}. Using baseline HCI/NLP dictionary.")
    VOCAB_WORDS = {"hello", "silent", "speech", "mobile", "recognizer", "phd", "research", "paper"}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"MELDER Execution Device: {device} | Dictionary size: {len(VOCAB_WORDS)} words")

NLTK corpus load warning: 
**********************************************************************
  Resource 'words' not found.
  Please use the NLTK Downloader to obtain the resource:

  >>> import nltk
  >>> nltk.download('words')

  For more information see: https://www.nltk.org/data.html

  Attempted to load 'corpora/words'

  Searched in:
    - '/root/nltk_data'
    - '/usr/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/share/nltk_data'
    - '/usr/local/share/nltk_data'
    - '/usr/lib/nltk_data'
    - '/usr/local/lib/nltk_data'
**********************************************************************
. Using baseline HCI/NLP dictionary.
MELDER Execution Device: cuda | Dictionary size: 8 words


[nltk_data] Error loading words: Security Violation [pathsec.urlopen]:
[nltk_data]     refusing a proxied fetch of
[nltk_data]     'https://raw.githubusercontent.com/nltk/nltk_data/gh-
[nltk_data]     pages/index.xml'. A configured proxy performs the
[nltk_data]     egress, so NLTK cannot pin the validated IP and SSRF
[nltk_data]     protection cannot be enforced (CWE-918). If and only
[nltk_data]     if the proxy is trusted to be SSRF-safe, opt in via
[nltk_data]     NLTK_ALLOW_PROXIED_URLOPEN=1 or
[nltk_data]     nltk.pathsec.ALLOW_PROXIED_FETCH=True.


In [38]:
# [CELL 2 - Optimized OpenCV Lip Extractor & Overlapping Slicer]

class LipLandmarkExtractor:
    """
    High-speed Lip ROI extractor using OpenCV Haar Cascades with image downscaling
    and temporal bounding-box caching to eliminate detection latency.
    """
    def __init__(self, target_size: Tuple[int, int] = (64, 64), detect_interval: int = 5):
        self.target_size = target_size
        self.detect_interval = detect_interval
        self.frame_count = 0
        self.cached_roi_coords = None

        # Load built-in OpenCV Haar Cascades
        self.face_cascade = cv2.CascadeClassifier(
            cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
        )

    def process_frame(self, frame: np.ndarray) -> np.ndarray:
        """Extracts normalized lip ROI with downscaled face detection & caching."""
        h, w, _ = frame.shape
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Run detection only on interval boundaries to minimize CPU load
        if self.frame_count % self.detect_interval == 0 or self.cached_roi_coords is None:
            # Downscale image by 0.25x for fast Haar detection
            scale = 0.25
            small_gray = cv2.resize(gray, (0, 0), fx=scale, fy=scale)

            faces = self.face_cascade.detectMultiScale(
                small_gray, scaleFactor=1.2, minNeighbors=4, minSize=(20, 20)
            )

            if len(faces) > 0:
                # Pick primary face and rescale coordinates back to full resolution
                s_fx, s_fy, s_fw, s_fh = max(faces, key=lambda rect: rect[2] * rect[3])
                fx, fy, fw, fh = int(s_fx / scale), int(s_fy / scale), int(s_fw / scale), int(s_fh / scale)

                # Derive lower face/mouth ROI
                mouth_y = fy + int(fh * 0.55)
                mouth_h = int(fh * 0.40)
                mouth_x = fx + int(fw * 0.15)
                mouth_w = int(fw * 0.70)

                self.cached_roi_coords = (
                    max(0, mouth_y), min(h, mouth_y + mouth_h),
                    max(0, mouth_x), min(w, mouth_x + mouth_w)
                )
            else:
                # Spatial fallback: Center-lower crop region
                cy, cx = int(h * 0.65), w // 2
                half_w, half_h = int(w * 0.15), int(h * 0.10)
                self.cached_roi_coords = (
                    max(0, cy - half_h), min(h, cy + half_h),
                    max(0, cx - half_w), min(w, cx + half_w)
                )

        self.frame_count += 1

        # Crop using cached ROI coordinates
        y1, y2, x1, x2 = self.cached_roi_coords
        lip_crop = gray[y1:y2, x1:x2]

        if lip_crop.size > 0:
            return cv2.resize(lip_crop, self.target_size) / 255.0

        return cv2.resize(gray, self.target_size) / 255.0


class OverlappingVideoSlicer:
    """
    Slices stream into overlapping temporal windows.
    """
    def __init__(self, window_size: int = 15, stride: int = 5):
        self.window_size = window_size
        self.stride = stride
        self.frame_buffer = deque(maxlen=window_size)
        self.counter = 0

    def add_frame(self, lip_frame: np.ndarray) -> bool:
        self.frame_buffer.append(lip_frame)
        self.counter += 1

        if len(self.frame_buffer) < self.window_size:
            return False

        return (self.counter - self.window_size) % self.stride == 0

    def get_clip(self) -> np.ndarray:
        return np.array(self.frame_buffer)

In [39]:
# [CELL 3 - MELDER Neural Network Architecture]

class LipReadingBackbone(nn.Module):
    """
    Spatiotemporal 3D-CNN + 2D ResNet Trunk + BiGRU for frame-level logit predictions.
    """
    def __init__(self, num_classes: int = 28):
        super(LipReadingBackbone, self).__init__()

        self.frontend = nn.Sequential(
            nn.Conv3d(1, 32, kernel_size=(3, 5, 5), stride=(1, 2, 2), padding=(1, 2, 2)),
            nn.BatchNorm3d(32),
            nn.PReLU(),
            nn.MaxPool3d(kernel_size=(1, 2, 2), stride=(1, 2, 2))
        )

        self.trunk = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.PReLU(),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.PReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.gru = nn.GRU(
            input_size=128,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.unsqueeze(1) # (B, 1, T, H, W)
        b, c, t, h, w = x.shape

        feat = self.frontend(x)
        feat = feat.permute(0, 2, 1, 3, 4).contiguous().view(b * t, 32, feat.shape[3], feat.shape[4])
        feat = self.trunk(feat)
        feat = feat.view(b, t, -1)

        gru_out, _ = self.gru(feat)
        logits = self.fc(gru_out)
        return logits

VOCAB = ["<blank>"] + [chr(i) for i in range(ord('a'), ord('z')+1)] + [" "]
model = LipReadingBackbone(num_classes=len(VOCAB)).to(device)
print(f"MELDER Neural Backbone loaded. Vocabulary classes: {len(VOCAB)}")

MELDER Neural Backbone loaded. Vocabulary classes: 28


In [40]:
# [CELL 4 - Overlap-Aware Stream Decoder & Auto-Completion]

class MELDERStreamDecoder:
    """
    Decodes CTC logits and uses suffix-prefix string matching to deduplicate
    overlapping sliding-window outputs.
    """
    def __init__(self, vocab: List[str], dictionary_set: set):
        self.vocab = vocab
        self.char_buffer = ""
        self.dictionary = dictionary_set
        self.dictionary.update(["hello", "silent", "speech", "mobile", "recognizer", "phd", "research"])

    def decode_logits(self, logits: torch.Tensor) -> str:
        """Standard CTC greedy decoding."""
        preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().numpy()
        decoded_chars = []
        prev_char_idx = 0

        for p in preds:
            if p != prev_char_idx and p != 0:  # Skip CTC blanks (0) and consecutive duplicates
                decoded_chars.append(self.vocab[p])
            prev_char_idx = p

        return "".join(decoded_chars)

    def update_buffer_and_autocomplete(self, raw_chars: str) -> Tuple[str, List[str]]:
        """Merges prediction using suffix-prefix overlap matching."""
        if raw_chars:
            # Find maximum common overlap between buffer suffix and raw_chars prefix
            max_overlap = 0
            max_search_len = min(len(self.char_buffer), len(raw_chars))

            for i in range(1, max_search_len + 1):
                if self.char_buffer[-i:] == raw_chars[:i]:
                    max_overlap = i

            # Append only non-overlapping trailing characters
            new_text = raw_chars[max_overlap:]
            self.char_buffer += new_text

        current_word = self.char_buffer.split()[-1] if self.char_buffer else ""

        suggestions = []
        if current_word:
            suggestions = [w for w in self.dictionary if w.lower().startswith(current_word.lower())][:3]

        return self.char_buffer, suggestions

    def reset_buffer(self):
        self.char_buffer = ""

In [41]:
# [CELL 5 - Full Pipeline Execution & Latency Benchmark]

def run_melder_pipeline(synthetic_video_stream: List[np.ndarray]):
    extractor = LipLandmarkExtractor(target_size=(64, 64))
    slicer = OverlappingVideoSlicer(window_size=15, stride=5)
    decoder = MELDERStreamDecoder(vocab=VOCAB, dictionary_set=VOCAB_WORDS)

    model.eval()
    print("\n--- Running Real-Time MELDER Silent Speech Inference ---")
    start_time = time.time()

    processed_clips = 0
    with torch.no_grad():
        for frame_idx, raw_frame in enumerate(synthetic_video_stream):
            lip_roi = extractor.process_frame(raw_frame)
            is_ready = slicer.add_frame(lip_roi)

            if is_ready:
                clip = slicer.get_clip()
                clip_tensor = torch.tensor(clip, dtype=torch.float32).unsqueeze(0).to(device)

                logits = model(clip_tensor)
                raw_text = decoder.decode_logits(logits)
                full_buffer, word_suggestions = decoder.update_buffer_and_autocomplete(raw_text)

                processed_clips += 1
                print(f"[Frame {frame_idx:03d} | Clip {processed_clips:02d}] "
                      f"Recognized Stream: '{full_buffer}' | "
                      f"Auto-Complete Candidates: {word_suggestions}")

    total_time = time.time() - start_time
    print(f"\n--- Execution Benchmark Summary ---")
    print(f"Total Frames Processed: {len(synthetic_video_stream)}")
    print(f"Total Temporal Window Inferences: {processed_clips}")
    print(f"Average Latency Per Window: {(total_time / max(1, processed_clips)) * 1000:.2f} ms")


# Generate synthetic video stream cleanly with torch tensors
dummy_frames = [
    torch.randint(0, 256, (480, 640, 3), dtype=torch.uint8).numpy()
    for _ in range(60)
]

# Run Pipeline
run_melder_pipeline(dummy_frames)


--- Running Real-Time MELDER Silent Speech Inference ---
[Frame 014 | Clip 01] Recognized Stream: 'x' | Auto-Complete Candidates: []
[Frame 019 | Clip 02] Recognized Stream: 'x' | Auto-Complete Candidates: []
[Frame 024 | Clip 03] Recognized Stream: 'x' | Auto-Complete Candidates: []
[Frame 029 | Clip 04] Recognized Stream: 'x' | Auto-Complete Candidates: []
[Frame 034 | Clip 05] Recognized Stream: 'x' | Auto-Complete Candidates: []
[Frame 039 | Clip 06] Recognized Stream: 'x' | Auto-Complete Candidates: []
[Frame 044 | Clip 07] Recognized Stream: 'x' | Auto-Complete Candidates: []
[Frame 049 | Clip 08] Recognized Stream: 'x' | Auto-Complete Candidates: []
[Frame 054 | Clip 09] Recognized Stream: 'x' | Auto-Complete Candidates: []
[Frame 059 | Clip 10] Recognized Stream: 'x' | Auto-Complete Candidates: []

--- Execution Benchmark Summary ---
Total Frames Processed: 60
Total Temporal Window Inferences: 10
Average Latency Per Window: 10.68 ms
